EE5121 - Convex Optimization  
Assignment 2 - Question 3

In [1]:
#imports
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from pathlib import Path

OUTDIR = Path(".")
OUTDIR.mkdir(exist_ok=True)

In [2]:
# Graphs:
V1 = [1,2,3,4]
edges1 = [(1,2),(1,3),(1,4),(2,3),(2,4),(3,4)]
n1=4
V2 = [1,2,3,4,5]
edges2 = [(1,2),(2,3),(3,4),(4,5),(5,1)] 
n2 = 5

In [3]:
#Build A matrix
def build_A_matrix(n, edges, alpha_vars):
    # alpha_vars: dict (u,v)-> cvxpy variable (or numeric)
    A = np.zeros((n,n), dtype=object)
    for i in range(n):
        for j in range(n):
            A[i,j] = 0.0
    for (u,v) in edges:
        # alpha index: (u-1,v-1)
        a = alpha_vars[(u,v)]
        A[u-1][v-1] = a
        A[v-1][u-1] = a
    # return cvxpy Expression matrix
    return cp.bmat([[cp.Constant(A[i][j]) if not isinstance(A[i][j], cp.Expression) else A[i][j] 
                     for j in range(n)] for i in range(n)])

In [4]:
#cvxpy solve wrapper for primal problem for reuse across graphs
def solve_primal(n, edges, solver=cp.ECOS):
    G = cp.Variable((n,n), PSD=True)
    rho = cp.Variable()
    cons = [cp.diag(G) == 1]
    for (u,v) in edges:
        cons.append(G[u-1, v-1] <= rho)
    prob = cp.Problem(cp.Minimize(rho), cons)
    prob.solve(solver=solver, verbose=False, eps=1e-8, max_iters=20000)
    return prob, G.value, float(rho.value) if rho.value is not None else None

In [5]:
# try ECOS then SCS
try:
    prob_k4, Gk4, rho_k4 = solve_primal(n1, edges1, solver=cp.ECOS)
    prob_c5, Gc5, rho_c5 = solve_primal(n2, edges2, solver=cp.ECOS)
    solver_used = "ECOS"
except Exception:
    prob_k4, Gk4, rho_k4 = solve_primal(n1, edges1, solver=cp.SCS)
    prob_c5, Gc5, rho_c5 = solve_primal(n2, edges2, solver=cp.SCS)
    solver_used = "SCS"

print("Primal solves done. Solver:", solver_used)
print("K4 primal status:", prob_k4.status, "rho* (primal) =", rho_k4)
print("C5 primal status:", prob_c5.status, "rho* (primal) =", rho_c5)


Primal solves done. Solver: SCS
K4 primal status: optimal rho* (primal) = -0.333333332557487
C5 primal status: optimal rho* (primal) = -0.809016994145889


In [6]:
#cvxpy solve wrapper for dual problem for reuse across graphs
def solve_dual(n, edges, solver=cp.ECOS):
    alpha = {}
    for (u,v) in edges:
        alpha[(u,v)] = cp.Variable(nonneg=True)
    lam = cp.Variable(n)   
    
    A = cp.Constant(np.zeros((n,n)))
    
    A_expr = cp.Constant(np.zeros((n,n)))
    
    A_entries = [[0 for _ in range(n)] for __ in range(n)]
    for i in range(n):
        for j in range(n):
            A_entries[i][j] = 0
    for (u,v) in edges:
        a = alpha[(u,v)]
        
        A_entries[u-1][v-1] = A_entries[v-1][u-1] = A_entries[u-1][v-1] + a
    
    A_mat = cp.bmat([[A_entries[i][j] if isinstance(A_entries[i][j], cp.Expression) else cp.Constant(float(A_entries[i][j]))
                      for j in range(n)] for i in range(n)])
    
    M = A_mat - cp.diag(lam)
    cons = [M >> 0, cp.sum([alpha[e] for e in edges]) == 1]
    
    prob = cp.Problem(cp.Maximize(cp.sum(lam)), cons)
    prob.solve(solver=solver, verbose=False, eps=1e-8, max_iters=20000)
    
    alpha_vals = {e: float(alpha[e].value) if alpha[e].value is not None else None for e in edges}
    lam_val = np.array(lam.value).ravel() if lam.value is not None else None
    return prob, lam_val, alpha_vals


In [7]:
#Try SCS if ECOS fails just as in the case of primal
try:
    dual_k4, lam_k4, alpha_k4 = solve_dual(n1, edges1, solver=cp.ECOS)
    dual_c5, lam_c5, alpha_c5 = solve_dual(n2, edges2, solver=cp.ECOS)
    dual_solver = "ECOS"
except Exception:
    dual_k4, lam_k4, alpha_k4 = solve_dual(n1, edges1, solver=cp.SCS)
    dual_c5, lam_c5, alpha_c5 = solve_dual(n2, edges2, solver=cp.SCS)
    dual_solver = "SCS"

# report
dval_k4 = None if lam_k4 is None else float(np.sum(lam_k4))
dval_c5 = None if lam_c5 is None else float(np.sum(lam_c5))

print("Dual solves done. Solver:", dual_solver)
print("K4 dual status:", dual_k4.status, "dual objective sum(lambda) =", dval_k4)
print("C5 dual status:", dual_c5.status, "dual objective sum(lambda) =", dval_c5)

# duality gaps
gap_k4 = None if (rho_k4 is None or dval_k4 is None) else abs(rho_k4 - dval_k4)
gap_c5 = None if (rho_c5 is None or dval_c5 is None) else abs(rho_c5 - dval_c5)

print("\nSummary:")
print("K4: primal rho* =", rho_k4, "dual d* =", dval_k4, "duality gap =", gap_k4)
print("C5: primal rho* =", rho_c5, "dual d* =", dval_c5, "duality gap =", gap_c5)


Dual solves done. Solver: SCS
K4 dual status: optimal dual objective sum(lambda) = -0.6666666668668381
C5 dual status: optimal dual objective sum(lambda) = -1.6180339877032068

Summary:
K4: primal rho* = -0.333333332557487 dual d* = -0.6666666668668381 duality gap = 0.3333333343093511
C5: primal rho* = -0.809016994145889 dual d* = -1.6180339877032068 duality gap = 0.8090169935573178
